In [1]:
import torch

# 从具体子模块导入，而不是 `from transformers import ...`：
# 后者会让编辑器的类型检查器解析整个 transformers（约 2500 个文件），非常慢
from transformers.models.qwen2 import Qwen2Tokenizer  # Qwen3 沿用 Qwen2 的 tokenizer
from transformers.models.qwen3 import Qwen3ForCausalLM

MODEL = "Qwen/Qwen3-0.6B"
# from_pretrained 的返回类型没有标注，手动标上，给编辑器补全用
tok: Qwen2Tokenizer = Qwen2Tokenizer.from_pretrained(MODEL)
model = Qwen3ForCausalLM.from_pretrained(MODEL)
model.eval()

TEMPERATURE = 0.001
print(model)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [11]:
prompt = "A tensor can be constructed from a Python list or sequence using the torch.tensor() constructor:"
print(f"Our prompt is: {prompt}")

encoded_prompt: list[int] = tok.encode(text=prompt)
print(f"The prompt will be encoded into tokens with ids: {encoded_prompt}")
prompt_tokens = tok.convert_ids_to_tokens(encoded_prompt)
print(f"And the ids represents: {prompt_tokens}")

Our prompt is: A tensor can be constructed from a Python list or sequence using the torch.tensor() constructor:
The prompt will be encoded into tokens with ids: [32, 15626, 646, 387, 20346, 504, 264, 13027, 1140, 476, 8500, 1667, 279, 7834, 40588, 368, 4692, 25]
And the ids represents: ['A', 'Ġtensor', 'Ġcan', 'Ġbe', 'Ġconstructed', 'Ġfrom', 'Ġa', 'ĠPython', 'Ġlist', 'Ġor', 'Ġsequence', 'Ġusing', 'Ġthe', 'Ġtorch', '.tensor', '()', 'Ġconstructor', ':']


In [14]:
vocabulary = tok.get_vocab()
print(len(vocabulary))
print(tok.all_special_tokens)

151669
['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']


In [37]:
embed = model.model.embed_tokens

WORDS = ["King", "Author", "Queen", "king", "shit", "dog"]
WORDS_VEC = []
for word in WORDS:
    WORDS_VEC.append(embed.weight[torch.tensor(tok.encode(word))])

for i in range(len(WORDS)):
    for j in range(i, len(WORDS)):
        print(f"The cosine similarity between `{WORDS[i]}` and `{WORDS[j]}` is: {torch.nn.functional.cosine_similarity(WORDS_VEC[i], WORDS_VEC[j])}")

The cosine similarity between `King` and `King` is: tensor([1.], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `King` and `Author` is: tensor([0.0894], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `King` and `Queen` is: tensor([0.4668], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `King` and `king` is: tensor([0.5156], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `King` and `shit` is: tensor([0.0815], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `King` and `dog` is: tensor([0.0718], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `Author` and `Author` is: tensor([1.], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `Author` and `Queen` is: tensor([0.1040], dtype=torch.bfloat16, grad_fn=<SumBackward1>)
The cosine similarity between `Author` and `king` is: tensor([0.0271], dtype=to

In [3]:
tok.get_vocab

<bound method TokenizersBackend.get_vocab of Qwen2Tokenizer(name_or_path='Qwen/Qwen3-0.6B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: A

In [1]:
from datasets import load_dataset

In [9]:
wiki_103 = load_dataset("iohadrubin/wikitext-103-raw-v1", split = "train")
print(f"numbers of lines of wiki_103: {len(wiki_103)}")
wiki_103[0]['text'][:200]

numbers of lines of wiki_103: 29567


'= Valkyria Chronicles III =\nSenjō no Valkyria 3: Unrecorded Chronicles (Japanese: 戦場のヴァルキュリア3, lit. Valkyria of the Battlefield 3), commonly referred to as Valkyria Chronicles III outside Japan, is a '

In [ ]:
appeared_tokens = set()
for i in range(100):
    encoded_tokens = tok.encode()